# Phase 2.5: Feature Reduction
**DNA Gene Mapping Project - ML Phase**  
**Author:** Sharique Mohammad  
**Date:** February 2026

## Objective
Apply all filters from Phase 2.1-2.4 and create final clean feature sets

## Key Tasks
1. Consolidate all exclusion criteria
2. Create final feature lists for each table
3. Validate feature sets are ML-ready
4. Export clean feature lists for Phase 3
5. Generate comprehensive Phase 2 summary report

## Deliverables
- Final feature lists (CSV) for each table
- Feature reduction summary report
- Phase 2 complete summary
- Ready-to-use feature sets for modeling

## Setup

In [ ]:
# Imports
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from pathlib import Path
import os
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path().absolute().parent.parent
REPORTS_DIR = PROJECT_ROOT / 'data' / 'analytical' / 'reports'
FEATURE_LISTS_DIR = PROJECT_ROOT / 'data' / 'analytical' / 'feature_lists'
FEATURE_LISTS_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(42)

print("Setup complete")
print(f"Reports: {REPORTS_DIR}")
print(f"Feature Lists: {FEATURE_LISTS_DIR}")

In [ ]:
# Database connection
load_dotenv()

POSTGRES_HOST = os.getenv('POSTGRES_HOST', 'localhost')
POSTGRES_PORT = os.getenv('POSTGRES_PORT', '5432')
POSTGRES_DB = os.getenv('POSTGRES_DB', 'genome_db')
POSTGRES_USER = os.getenv('POSTGRES_USER', 'postgres')
POSTGRES_PASSWORD = os.getenv('POSTGRES_PASSWORD')

conn_str = f"postgresql://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"
engine = create_engine(conn_str)

print("Database connection established")

## 1. Load All Table Schemas

In [ ]:
# Load column names for each table
print("Loading table schemas...\n")

tables = [
    'clinical_ml_features',
    'disease_ml_features',
    'pharmacogene_ml_features',
    'variant_impact_ml_features',
    'structural_variant_ml_features'
]

table_columns = {}

for table_name in tables:
    query = f"SELECT * FROM gold.{table_name} LIMIT 0"
    df = pd.read_sql(query, engine)
    table_columns[table_name] = df.columns.tolist()
    print(f"{table_name}: {len(df.columns)} columns")

print(f"\nLoaded schemas for {len(table_columns)} tables")

## 2. Load Previous Phase Exclusions

In [ ]:
# Attempt to load exclusion lists from previous phases
print("Loading exclusion lists from previous phases...\n")

exclusions_from_previous = {}

# Try to load leakage exclusions from Phase 2.4
for table_name in tables:
    exclusion_file = REPORTS_DIR / f"{table_name}_exclusions.csv"
    
    if exclusion_file.exists():
        exclusion_df = pd.read_csv(exclusion_file)
        if 'excluded_feature' in exclusion_df.columns:
            exclusions_from_previous[table_name] = set(exclusion_df['excluded_feature'].tolist())
            print(f"{table_name}: {len(exclusions_from_previous[table_name])} exclusions loaded")
        else:
            print(f"{table_name}: Exclusion file found but no excluded_feature column")
    else:
        print(f"{table_name}: No exclusion file found")
        exclusions_from_previous[table_name] = set()

total_exclusions = sum(len(v) for v in exclusions_from_previous.values())
print(f"\nTotal exclusions loaded: {total_exclusions}")

## 3. Define Manual Exclusion Rules

In [ ]:
# Define additional manual exclusion patterns
# These are features that should always be excluded

print("Applying manual exclusion rules...\n")

manual_exclusion_patterns = [
    # IDs and identifiers
    'variant_id', 'sv_id', 'study_id',
    
    # Positions and locations
    'position', 'start_pos', 'end_pos', 'chromosome',
    
    # Names and descriptions
    'gene_name', 'variant_name', 'official_symbol', 'validated_gene_symbol',
    'official_gene_symbol', 'gene_description', 'assembly',
    
    # Target leakage (absolute must remove)
    'target_is_pathogenic', 'target_is_benign', 'target_is_vus',
    'is_pathogenic', 'is_benign', 'is_vus',
    'clinical_significance_simple', 'clinvar_pathogenicity_class',
    'predicted_sv_pathogenicity', 'is_high_risk_sv',
    
    # Text fields (can't use directly in ML)
    'protein_change', 'cdna_change', 'variant_type',
    'affected_genes', 'complete_overlap_genes', 'major_overlap_genes',
    'pharmacogenes_affected', 'kinases_affected', 'receptors_affected',
    'omim_genes_affected', 'genes_lost', 'genes_gained'
]

manual_exclusions = {}

for table_name, columns in table_columns.items():
    excluded = set()
    
    for col in columns:
        if col in manual_exclusion_patterns:
            excluded.add(col)
    
    if excluded:
        manual_exclusions[table_name] = excluded
        print(f"{table_name}: {len(excluded)} manual exclusions")

total_manual = sum(len(v) for v in manual_exclusions.values())
print(f"\nTotal manual exclusions: {total_manual}")

## 4. Consolidate All Exclusions

In [ ]:
# Combine all exclusion sources
print("Consolidating all exclusions...\n")

final_exclusions = {}

for table_name in tables:
    all_excluded = set()
    
    # Add exclusions from previous phases
    if table_name in exclusions_from_previous:
        all_excluded.update(exclusions_from_previous[table_name])
    
    # Add manual exclusions
    if table_name in manual_exclusions:
        all_excluded.update(manual_exclusions[table_name])
    
    final_exclusions[table_name] = all_excluded
    
    print(f"{table_name}:")
    print(f"  From previous phases: {len(exclusions_from_previous.get(table_name, set()))}")
    print(f"  Manual exclusions: {len(manual_exclusions.get(table_name, set()))}")
    print(f"  Total to exclude: {len(all_excluded)}")
    print()

total_final_exclusions = sum(len(v) for v in final_exclusions.values())
print(f"Total features to exclude across all tables: {total_final_exclusions}")

## 5. Create Final Clean Feature Lists

In [ ]:
# Create final feature lists after all exclusions
print("Creating final clean feature lists...\n")

final_feature_lists = {}

for table_name, all_columns in table_columns.items():
    excluded = final_exclusions.get(table_name, set())
    
    # Keep features that are NOT in exclusion list
    clean_features = [col for col in all_columns if col not in excluded]
    
    final_feature_lists[table_name] = clean_features
    
    print(f"{table_name}:")
    print(f"  Original columns: {len(all_columns)}")
    print(f"  Excluded: {len(excluded)}")
    print(f"  Clean features: {len(clean_features)}")
    print(f"  Retention rate: {len(clean_features)/len(all_columns)*100:.1f}%")
    print()

total_clean = sum(len(v) for v in final_feature_lists.values())
total_original = sum(len(v) for v in table_columns.values())

print(f"Overall:")
print(f"  Original columns: {total_original}")
print(f"  Excluded: {total_final_exclusions}")
print(f"  Clean features: {total_clean}")
print(f"  Overall retention: {total_clean/total_original*100:.1f}%")

## 6. Validate Feature Lists

In [ ]:
# Validate that critical features were NOT removed
print("Validating feature lists...\n")

validation_passed = True

# Check that we have enough features for each table
for table_name, features in final_feature_lists.items():
    if len(features) < 5:
        print(f"WARNING: {table_name} has only {len(features)} features!")
        validation_passed = False
    else:
        print(f"{table_name}: {len(features)} features (OK)")

# Check that target columns were removed
target_columns = ['target_is_pathogenic', 'target_is_benign', 'is_high_risk_sv']
print("\nVerifying target columns were removed:")
for table_name, features in final_feature_lists.items():
    for target in target_columns:
        if target in features:
            print(f"  ERROR: {table_name} still contains {target}!")
            validation_passed = False

if validation_passed:
    print("  PASS: All target columns removed")

# Check that ID columns were removed
id_columns = ['variant_id', 'sv_id']
print("\nVerifying ID columns were removed:")
for table_name, features in final_feature_lists.items():
    for id_col in id_columns:
        if id_col in features:
            print(f"  ERROR: {table_name} still contains {id_col}!")
            validation_passed = False

if validation_passed:
    print("  PASS: All ID columns removed")

print("\n" + "="*80)
if validation_passed:
    print("VALIDATION PASSED: Feature lists are ready for modeling")
else:
    print("VALIDATION FAILED: Review warnings above")
print("="*80)

## 7. Export Final Feature Lists

In [ ]:
# Export final feature lists as CSV files
print("\nExporting final feature lists...\n")

for table_name, features in final_feature_lists.items():
    # Create DataFrame with feature names and metadata
    feature_df = pd.DataFrame({
        'feature_name': features,
        'table': table_name,
        'feature_index': range(len(features))
    })
    
    # Save to CSV
    filename = f"{table_name}_final_features.csv"
    feature_df.to_csv(FEATURE_LISTS_DIR / filename, index=False)
    print(f"Saved: {FEATURE_LISTS_DIR / filename} ({len(features)} features)")

# Export combined feature list
all_features = []
for table_name, features in final_feature_lists.items():
    for feat in features:
        all_features.append({
            'table': table_name,
            'feature_name': feat
        })

combined_df = pd.DataFrame(all_features)
combined_df.to_csv(FEATURE_LISTS_DIR / 'all_final_features.csv', index=False)
print(f"\nSaved: {FEATURE_LISTS_DIR / 'all_final_features.csv'} ({len(all_features)} total features)")

## 8. Generate Summary Statistics

In [ ]:
# Create comprehensive summary
summary_data = []

for table_name in tables:
    original = len(table_columns[table_name])
    excluded = len(final_exclusions.get(table_name, set()))
    final = len(final_feature_lists[table_name])
    
    summary_data.append({
        'Table': table_name.replace('_ml_features', ''),
        'Original': original,
        'Excluded': excluded,
        'Final': final,
        'Retention %': f"{final/original*100:.1f}%"
    })

summary_df = pd.DataFrame(summary_data)

print("\nFeature Reduction Summary:")
print("="*80)
print(summary_df.to_string(index=False))
print("="*80)

print(f"\nTotals:")
print(f"  Original features: {summary_df['Original'].sum()}")
print(f"  Excluded features: {summary_df['Excluded'].sum()}")
print(f"  Final clean features: {summary_df['Final'].sum()}")
print(f"  Overall retention: {summary_df['Final'].sum()/summary_df['Original'].sum()*100:.1f}%")

## 9. Generate Phase 2 Complete Report

In [ ]:
# Generate comprehensive Phase 2 summary report
report_path = REPORTS_DIR / 'phase2_feature_selection_complete.txt'

with open(report_path, 'w') as f:
    f.write("="*80 + "\n")
    f.write("PHASE 2 COMPLETE: FEATURE SELECTION\n")
    f.write("DNA Gene Mapping Project\n")
    f.write("="*80 + "\n\n")
    
    f.write("PHASE 2 SUMMARY\n")
    f.write("-"*80 + "\n")
    f.write("Completed 5 notebooks:\n")
    f.write("  2.1 Target Definition\n")
    f.write("  2.2 Feature Quality Checks\n")
    f.write("  2.3 Correlation Analysis\n")
    f.write("  2.4 Leakage Detection\n")
    f.write("  2.5 Feature Reduction\n\n")
    
    f.write("FINAL FEATURE COUNTS\n")
    f.write("-"*80 + "\n")
    f.write(summary_df.to_string(index=False))
    f.write("\n\n")
    
    f.write("OVERALL STATISTICS\n")
    f.write("-"*80 + "\n")
    f.write(f"Original features: {summary_df['Original'].sum()}\n")
    f.write(f"Excluded features: {summary_df['Excluded'].sum()}\n")
    f.write(f"Final clean features: {summary_df['Final'].sum()}\n")
    f.write(f"Overall retention: {summary_df['Final'].sum()/summary_df['Original'].sum()*100:.1f}%\n\n")
    
    f.write("EXCLUSION BREAKDOWN\n")
    f.write("-"*80 + "\n")
    f.write("Features excluded due to:\n")
    f.write("  - Target leakage (critical)\n")
    f.write("  - High missing values (>80%)\n")
    f.write("  - Zero variance\n")
    f.write("  - High correlation (>0.95)\n")
    f.write("  - ID/metadata columns\n")
    f.write("  - Text fields (non-numeric)\n\n")
    
    f.write("FINAL FEATURE LISTS\n")
    f.write("="*80 + "\n")
    for table_name, features in final_feature_lists.items():
        f.write(f"\n{table_name} ({len(features)} features):\n")
        for feat in sorted(features):
            f.write(f"  - {feat}\n")
    
    f.write("\n" + "="*80 + "\n")
    f.write("DELIVERABLES\n")
    f.write("="*80 + "\n")
    f.write("Files created:\n")
    f.write(f"  - {len(final_feature_lists)} table-specific feature lists\n")
    f.write("  - 1 combined feature list (all tables)\n")
    f.write(f"  - Location: {FEATURE_LISTS_DIR}\n\n")
    
    f.write("VALIDATION STATUS\n")
    f.write("-"*80 + "\n")
    if validation_passed:
        f.write("PASSED: All validation checks successful\n")
        f.write("  - Target columns removed\n")
        f.write("  - ID columns removed\n")
        f.write("  - Sufficient features per table\n")
    else:
        f.write("FAILED: Review validation warnings\n")
    f.write("\n")
    
    f.write("READY FOR PHASE 3: MODELING\n")
    f.write("="*80 + "\n")
    f.write("Next steps:\n")
    f.write("  1. Load feature lists from CSV\n")
    f.write("  2. Create train/validation/test splits\n")
    f.write("  3. Handle class imbalance (3.90:1 variants, 26:1 SVs)\n")
    f.write("  4. Train baseline models\n")
    f.write("  5. Hyperparameter tuning\n")
    f.write("  6. Model evaluation and selection\n")

print(f"\nPhase 2 complete report saved: {report_path}")
print("\n" + "="*80)
print("PHASE 2 COMPLETE - FEATURE SELECTION")
print("="*80)
print(f"\nFinal Statistics:")
print(f"  Clean features ready for modeling: {summary_df['Final'].sum()}")
print(f"  Feature lists exported to: {FEATURE_LISTS_DIR}")
print(f"\nPhase 2 deliverables:")
print(f"  - Target definitions")
print(f"  - Quality checks report")
print(f"  - Correlation analysis")
print(f"  - Leakage detection")
print(f"  - Final feature lists (CSV)")
print("\nREADY FOR PHASE 3: MODELING")